# 3. Training the ISL sign recognition model

## Key concepts

- **Training data**: example landmark sequences + their correct labels.
- **Label**: the correct answer, e.g. `HELLO`.
- **Features**: the numbers we feed the model — landmark coordinates.
- **Classification**: predicting which sign an input belongs to.
- **Epoch**: one full pass through all training data.
- **Loss**: how wrong the predictions are (lower = better).
- **Optimizer (Adam)**: nudges the model's numbers to reduce loss.

## Why LSTM

```text
Landmark sequence -> LSTM -> Linear layer -> Sign prediction
```

Sign language is movement over time, not a single photo. An LSTM reads a
sequence step by step and keeps a running "memory", which is what's needed
to tell HELLO's motion apart from THANK_YOU's motion.

## 0. Setup (Colab or local)

On Colab, turn on a free GPU first: Runtime > Change runtime type > T4 GPU.

In [ ]:
import os, json, copy, glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/ISL_Sign2Sub"
    ON_COLAB = True
except ImportError:
    PROJECT_DIR = os.path.abspath("..")
    ON_COLAB = False

LANDMARKS_DIR = os.path.join(PROJECT_DIR, "data", "landmarks")
MODEL_DIR = os.path.join(PROJECT_DIR, "model")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Running on Colab:", ON_COLAB)
print("Torch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if ON_COLAB and device.type != "cuda":
    print("Tip: Runtime > Change runtime type > GPU, then rerun this cell.")

## 1. Load all landmark sequences and their labels

The label for each video is taken from its filename, e.g.
`HELLO__MVI_0029.npy` -> label `HELLO`. Class numbers are assigned in
alphabetical order so the mapping is stable.

In [ ]:
files = sorted(glob.glob(os.path.join(LANDMARKS_DIR, "*.npy")))

sequences, labels = [], []
for f in files:
    sequences.append(np.load(f))
    labels.append(os.path.basename(f).split("__")[0])

class_names_list = sorted(set(labels))
class_to_idx = {name: i for i, name in enumerate(class_names_list)}

print("Total videos:", len(sequences))
print("Number of classes:", len(class_names_list))
for name, idx in class_to_idx.items():
    print(f"  {idx} -> {name}  ({labels.count(name)} videos)")

lengths = [s.shape[0] for s in sequences]
print("\nSequence length (frames): min", min(lengths), "| max", max(lengths),
      "| average %.0f" % (sum(lengths) / len(lengths)))

## 2. Resample to a fixed length and split train/val/test

Videos have different frame counts, but the LSTM needs equal-length
sequences. Instead of padding with zeros (which dilutes the LSTM's memory
with fake frames), we **resample**: pick `TARGET_LEN` evenly spaced frames
from each video, so short and long videos both become the same length while
still covering the whole sign.

Split: 70% train / 15% validation / 15% test, stratified so each sign is
represented proportionally in all three sets.

In [ ]:
TARGET_LEN = 64

def resample_sequence(seq, target_len=TARGET_LEN):
    idx = np.linspace(0, len(seq) - 1, target_len).round().astype(int)
    return seq[idx]

X = np.stack([resample_sequence(s) for s in sequences]).astype(np.float32)
y = np.array([class_to_idx[l] for l in labels])
print("X shape:", X.shape, "(videos, frames, numbers per frame)")

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print("Train:", X_train.shape[0], "| Validation:", X_val.shape[0], "| Test:", X_test.shape[0])

**Honest note:** videos come in consecutive takes (e.g. MVI_0029, MVI_0030)
that are likely the same signer repeating the sign. A random split can put
near-identical takes on both sides, making validation/test accuracy
somewhat optimistic. The real check is testing on a video the model has
never seen (notebook 4).

## 3. Normalize landmarks relative to the shoulders

Raw landmark coordinates encode where the signer stands in the frame, not
just the sign itself — a model trained on raw coordinates has to separately
learn to ignore camera position/distance, which wastes capacity and hurts
accuracy. We fix this by re-expressing every point relative to the
shoulders: subtract the shoulder midpoint, divide by shoulder width.

**Tenglish:** "Ippudu model ki person ekkada nilchunnado kuda ardham
chesukovali. Manam hands ni shoulders ki relative ga marchithe, person
position tho sambandham lekunda sign matrame migulthundi."

**This step must also happen at prediction time** (notebook 4, app.py) —
the saved model only works correctly on normalized input.

In [ ]:
def normalize_sequences(X):
    N, T, _ = X.shape
    pose = X[:, :, 126:].reshape(N, T, 33, 3)
    ls, rs = pose[:, :, 11, :], pose[:, :, 12, :]          # left/right shoulder
    center = (ls + rs) / 2
    scale = np.linalg.norm((ls - rs)[:, :, :2], axis=2, keepdims=True)
    scale = np.where(scale < 1e-3, 1.0, scale)             # avoid dividing by ~0

    out = X.copy()
    for start in (0, 63):                                  # left hand, right hand
        hand = X[:, :, start:start + 63].reshape(N, T, 21, 3)
        missing = np.all(hand == 0, axis=(2, 3), keepdims=True)
        hand_n = (hand - center[:, :, None, :]) / scale[:, :, None, :]
        out[:, :, start:start + 63] = np.where(missing, 0.0, hand_n).reshape(N, T, 63)
    pose_n = (pose - center[:, :, None, :]) / scale[:, :, None, :]
    out[:, :, 126:] = pose_n.reshape(N, T, 99)
    return out.astype(np.float32)

X_norm = normalize_sequences(X)
Xn_train, Xn_temp, y_train, y_temp = train_test_split(
    X_norm, y, test_size=0.30, stratify=y, random_state=42)
Xn_val, Xn_test, y_val, y_test = train_test_split(
    Xn_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

X_train_t = torch.tensor(Xn_train); y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t   = torch.tensor(Xn_val);   y_val_t   = torch.tensor(y_val,   dtype=torch.long)
X_test_t  = torch.tensor(Xn_test);  y_test_t  = torch.tensor(y_test,  dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=16)

input_size = X.shape[2]
num_classes = len(class_names_list)
hidden_size = 64
print("Normalized. input_size:", input_size, "num_classes:", num_classes)

## 4. Define the LSTM model

In [ ]:
class SignLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_classes=10):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]           # final memory state
        logits = self.fc(last_hidden)   # -> class scores
        return logits

model = SignLSTM(input_size, hidden_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))
print("Batches per epoch:", len(train_loader))

## 5. Train

An **epoch** is one full pass through the training videos. Gradient
clipping (`clip_grad_norm_`) prevents sudden unstable jumps in the loss. The
best-performing epoch's weights are kept, not necessarily the last one.

In [ ]:
EPOCHS = 100
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc, best_epoch, best_state = 0.0, 0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    tot_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tot_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    train_loss, train_acc = tot_loss / total, correct / total

    model.eval()
    tot_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            tot_loss += criterion(logits, yb).item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    val_loss, val_acc = tot_loss / total, correct / total

    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc, best_epoch = val_acc, epoch
        best_state = copy.deepcopy(model.state_dict())

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train loss {train_loss:.3f} acc {train_acc:.1%} "
              f"| val loss {val_loss:.3f} acc {val_acc:.1%}")

print(f"\nBest validation accuracy: {best_val_acc:.1%} (epoch {best_epoch})")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["train_loss"], label="Train"); ax[0].plot(history["val_loss"], label="Validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history["train_acc"], label="Train"); ax[1].plot(history["val_acc"], label="Validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.show()

**Reading the graphs:** both loss lines falling means the model is
learning. If training accuracy is much higher than validation accuracy,
that's overfitting — expected to some degree with a small dataset. Rising
validation loss over many epochs would be a bad sign; a gap between train
and validation accuracy alone is normal and mild overfitting, not broken.

## 6. Evaluate on the test set

The test set was never used for training or for picking the best epoch, so
this is the most honest accuracy number — quote this one, not the
validation number above.

In [ ]:
model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    logits = model(X_test_t.to(device))
    preds = logits.argmax(1).cpu().numpy()

test_acc = (preds == y_test).mean()
print(f"Test accuracy: {test_acc:.1%} ({(preds == y_test).sum()}/{len(y_test)} correct)")

print("\nWrong predictions:")
for true, pred in zip(y_test, preds):
    if true != pred:
        print(f"  actual {class_names_list[true]} -> predicted {class_names_list[pred]}")

## 7. Save the model to Drive

`target_len` and `normalize` are saved alongside the weights so that
notebook 4 and `app.py` preprocess new video identically to training —
skipping either step produces wrong predictions, since the model only
understands normalized, 64-frame input.

In [ ]:
torch.save(
    {
        "state_dict": best_state,
        "input_size": input_size,
        "hidden_size": hidden_size,
        "target_len": TARGET_LEN,
        "normalize": "shoulder",
    },
    os.path.join(MODEL_DIR, "sign_model.pth"),
)

class_names = {str(i): name for i, name in enumerate(class_names_list)}
with open(os.path.join(MODEL_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f, indent=2)

print("Saved to:", MODEL_DIR)
print(os.listdir(MODEL_DIR))